# Week 4-5: Gradient Descent

## What We're Learning

Last week we understood what a **weight** is and how a prediction is made:  
`ŷ = w × z + b`

This week we answer the most important question in all of machine learning:

> **How does the model know which direction to adjust its weights?**

The answer is **gradient descent** — the engine behind almost every ML model you will ever use.

---

## The Core Idea: The Error Landscape

Imagine you are standing on a hilly landscape. Your position on the horizontal axis represents the current value of weight `w`. The height you are standing at represents the **error** — how wrong your model is.

- At the **bottom of the valley** → error is 0, perfect predictions
- **High up on the hill** → large error, bad predictions

Your goal: **walk downhill to reach the bottom.**

How do you know which direction is downhill? You look at the **slope** of the ground under your feet. That slope is called the **gradient**.

---

## The Three Key Formulas

| Formula | Meaning |
|---|---|
| `L = (ŷ - y)²` | Loss (how wrong the prediction is) |
| `∂L/∂w = 2(ŷ - y) × z` | Gradient (slope of loss with respect to weight `w`) |
| `w_new = w - lr × gradient` | Weight update (take one step downhill) |

**Why subtract the gradient?**  
If the slope is positive (going uphill to the right), we move **left** (decrease `w`).  
If the slope is negative (going downhill to the right), we move **right** (increase `w`).  
Subtracting the gradient always moves us toward the valley.

**Learning rate `lr`:** Controls how big a step we take. Too big → overshoot. Too small → too slow.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── Global style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#f9f9f9',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
})

# ─── Helper: compute loss landscape and mark a point ─────────────────────────
def plot_landscape(ax, w_current, y, z, title, color='steelblue', lr=0.1):
    """
    Draw the parabolic error landscape L(w) = (w*z - y)^2,
    mark the current position, the tangent (gradient arrow), and the optimum.
    """
    # Range of w values to plot
    w_opt = y / z if z != 0 else 0        # where loss = 0
    lo = min(w_current, w_opt) - 1.5
    hi = max(w_current, w_opt) + 1.5
    w_range = np.linspace(lo, hi, 300)
    loss_range = (w_range * z - y) ** 2

    ax.plot(w_range, loss_range, color=color, lw=2, label='Loss curve')

    # Current position
    y_hat = w_current * z
    loss_now = (y_hat - y) ** 2
    grad     = 2 * (y_hat - y) * z

    ax.scatter([w_current], [loss_now], color='red', zorder=5, s=80, label=f'Current w={w_current:.2f}')

    # Gradient arrow (tangent direction, scaled for visibility)
    arrow_scale = 0.4
    ax.annotate(
        '', 
        xy=(w_current - arrow_scale * np.sign(grad), loss_now - arrow_scale * abs(grad) * 0.5),
        xytext=(w_current, loss_now),
        arrowprops=dict(arrowstyle='->', color='darkorange', lw=2)
    )
    ax.text(w_current + 0.05, loss_now + 0.05, f'grad={grad:.3f}', fontsize=9, color='darkorange')

    # Optimal point
    ax.scatter([w_opt], [0], color='green', zorder=5, s=80, marker='*', label=f'Optimal w={w_opt:.2f}')

    # Updated weight
    w_new = w_current - lr * grad
    loss_new = (w_new * z - y) ** 2
    ax.scatter([w_new], [loss_new], color='purple', zorder=5, s=60, marker='D', label=f'After update w={w_new:.3f}')

    ax.set_xlabel('Weight w')
    ax.set_ylabel('Loss L = (ŷ - y)²')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

print("Helper functions loaded. Ready for examples!")

---
# Example 1: Loan Approval (Baseline)

**Scenario:** A bank predicts whether to approve a loan.

| Variable | Value | Meaning |
|---|---|---|
| `y` | 1.0 | Ground truth: loan **should** be approved |
| `ŷ` | 0.15 | Model's prediction: only 15% confident — very wrong! |
| `z_age` | 1.5 | Applicant's age is 1.5 standard deviations above average |
| `w` | 0.1 | Current weight for age feature |
| `lr` | 0.1 | Learning rate |

**Step 1:** Compute the gradient  
`∂L/∂w = 2 × (ŷ - y) × z = 2 × (0.15 - 1.0) × 1.5`

**Step 2:** Update the weight  
`w_new = w - lr × gradient`

**What do we expect?**  
- `ŷ < y` so `(ŷ - y)` is **negative**  
- `z > 0` so gradient is **negative**  
- Subtracting a negative → weight **increases** → next prediction will be higher → closer to 1.0  

This is exactly right! To predict closer to 1, we need a bigger weight on age.

In [ ]:
# ─── Example 1: Loan Approval ────────────────────────────────────────────────
y_loan   = 1.0    # true label: approved
y_hat_1  = 0.15   # model prediction
z_age    = 1.5    # standardized age feature
w_1      = 0.1    # starting weight
lr_1     = 0.1

# Step 1: Loss before update
loss_before = (y_hat_1 - y_loan) ** 2

# Step 2: Gradient
grad_1 = 2 * (y_hat_1 - y_loan) * z_age

# Step 3: Update weight
w_new_1 = w_1 - lr_1 * grad_1

# Step 4: New prediction with updated weight
y_hat_new_1 = w_new_1 * z_age
loss_after  = (y_hat_new_1 - y_loan) ** 2

print("=" * 55)
print("EXAMPLE 1: LOAN APPROVAL")
print("=" * 55)
print(f"  True label      y  = {y_loan}")
print(f"  Prediction      ŷ  = {y_hat_1}")
print(f"  Feature       z_age = {z_age}")
print(f"  Current weight  w  = {w_1}")
print()
print(f"  Loss before:  L = ({y_hat_1} - {y_loan})² = {loss_before:.4f}")
print()
print(f"  Gradient = 2 × ({y_hat_1} - {y_loan}) × {z_age}")
print(f"           = 2 × ({y_hat_1 - y_loan:.2f}) × {z_age}")
print(f"           = {grad_1:.4f}  ← NEGATIVE (weight must increase)")
print()
print(f"  w_new = {w_1} - {lr_1} × ({grad_1:.4f})")
print(f"        = {w_1} - ({lr_1 * grad_1:.4f})")
print(f"        = {w_new_1:.4f}  ← weight increased ✓")
print()
print(f"  New prediction ŷ = {w_new_1:.4f} × {z_age} = {y_hat_new_1:.4f}")
print(f"  Loss after:   L = {loss_after:.4f}")
print(f"  Improvement:  {loss_before:.4f} → {loss_after:.4f}  (reduced by {loss_before - loss_after:.4f}) ✓")

In [ ]:
# ─── Visualize Example 1 error landscape ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
plot_landscape(ax, w_1, y_loan, z_age,
               title='Example 1 — Loan Approval\nError Landscape (one weight, one feature)',
               color='steelblue', lr=lr_1)
plt.tight_layout()
plt.show()

**Reading the plot:**
- The **red dot** is where we start — high up on the loss curve, far from the valley
- The **orange arrow** points in the direction the gradient is pushing us — downhill
- The **purple diamond** is where we land after one update — closer to the valley
- The **green star** is the optimal weight where loss = 0

One step alone doesn't get us there — we need many iterations. We'll see that in the convergence section.

---
# Example 2: House Price Prediction

**Scenario:** Predict whether a house price is above average (y=1) or below (y=0). Normalized output.

| Variable | Value | Meaning |
|---|---|---|
| `y` | 1.0 | Normalized true price (high-end house) |
| `ŷ` | 0.4 | Prediction: only 40% — prediction too low |
| `z_size` | 0.8 | House size is slightly above average |
| `w` | 0.5 | Current weight for size |
| `lr` | 0.1 | Learning rate |

**Key intuition:** `ŷ < y` means we are **underestimating**. The gradient will be negative, so the weight will increase. This makes sense — if larger houses cost more, the weight for size should be bigger.

In [ ]:
# ─── Example 2: House Price ───────────────────────────────────────────────────
y_house  = 1.0
y_hat_2  = 0.4
z_size   = 0.8
w_2      = 0.5
lr_2     = 0.1

loss_before_2 = (y_hat_2 - y_house) ** 2
grad_2        = 2 * (y_hat_2 - y_house) * z_size
w_new_2       = w_2 - lr_2 * grad_2
y_hat_new_2   = w_new_2 * z_size
loss_after_2  = (y_hat_new_2 - y_house) ** 2

print("=" * 55)
print("EXAMPLE 2: HOUSE PRICE")
print("=" * 55)
print(f"  (ŷ - y) = {y_hat_2} - {y_house} = {y_hat_2 - y_house:.1f}  ← NEGATIVE")
print(f"  z_size  = {z_size}  ← POSITIVE")
print(f"  Gradient = 2 × {y_hat_2 - y_house:.1f} × {z_size} = {grad_2:.4f}  ← NEGATIVE")
print(f"  w_new = {w_2} - {lr_2} × {grad_2:.4f} = {w_new_2:.4f}  ← INCREASED ✓")
print(f"  Loss:  {loss_before_2:.4f} → {loss_after_2:.4f}  (improved ✓)")
print()
print("  Interpretation: The model was underestimating the house price.")
print("  Gradient descent correctly pushes the weight UP, so future")
print("  predictions for large houses will be higher.")

---
# Example 3: Spam Detection

**Scenario:** Classify an email as spam (y=1) based on how many times the word "free" appears.

| Variable | Value | Meaning |
|---|---|---|
| `y` | 1.0 | Is spam |
| `ŷ` | 0.7 | Prediction: 70% likely spam — close but not enough |
| `z_free` | 1.2 | "free" count is 1.2 std above average |
| `w` | 0.583 | Current weight for "free" count |
| `lr` | 0.1 | Learning rate |

**Key intuition:** The prediction is fairly close (0.7 vs 1.0), so the error `(ŷ - y) = -0.3` is small. This means the gradient will be **small** and the weight update will be a **small step**. The model is learning efficiently — it already has the right direction, just needs fine-tuning.

In [ ]:
# ─── Example 3: Spam Detection ───────────────────────────────────────────────
y_spam   = 1.0
y_hat_3  = 0.7
z_free   = 1.2
w_3      = 0.583
lr_3     = 0.1

loss_before_3 = (y_hat_3 - y_spam) ** 2
grad_3        = 2 * (y_hat_3 - y_spam) * z_free
w_new_3       = w_3 - lr_3 * grad_3
y_hat_new_3   = w_new_3 * z_free
loss_after_3  = (y_hat_new_3 - y_spam) ** 2

print("=" * 55)
print("EXAMPLE 3: SPAM DETECTION")
print("=" * 55)
print(f"  Error   = ŷ - y = {y_hat_3} - {y_spam} = {y_hat_3 - y_spam:.1f}  (small negative)")
print(f"  Gradient = 2 × {y_hat_3 - y_spam:.1f} × {z_free} = {grad_3:.4f}  ← SMALL")
print(f"  Step size = lr × |gradient| = {lr_3} × {abs(grad_3):.4f} = {lr_3 * abs(grad_3):.4f}")
print(f"  w_new = {w_3} - ({lr_3 * grad_3:.4f}) = {w_new_3:.4f}")
print(f"  Loss:  {loss_before_3:.4f} → {loss_after_3:.4f}  (small improvement — expected ✓)")
print()
print("  Key lesson: Small errors → small gradients → small weight updates.")
print("  The model is 'confident but wrong in a minor way', so it adjusts gently.")

---
# Example 4: Student Grade — Weight Must DECREASE

**Scenario:** Predict whether a student fails (y=0) based on attendance.

| Variable | Value | Meaning |
|---|---|---|
| `y` | 0.0 | Student failed |
| `ŷ` | 0.6 | Prediction: 60% chance of passing — **very wrong!** |
| `z_attend` | -0.5 | Below average attendance |
| `w` | -1.2 | Current weight (negative = low attendance → fail) |
| `lr` | 0.1 | Learning rate |

**Key intuition:** The model predicts the student will pass, but they failed. The error `(ŷ - y) = +0.6` is **positive** (over-predicted). The feature `z_attend = -0.5` is **negative**.

- Gradient = 2 × (+0.6) × (-0.5) = **negative**
- We subtract a negative → weight **increases** (becomes less negative)

Wait — does this make sense? If weight becomes less negative, then for a student with negative attendance, the prediction `w × z_attend` becomes... less negative... so predictions go lower. That is correct! The model is learning that negative attendance more strongly predicts failure.

In [ ]:
# ─── Example 4: Student Grade ─────────────────────────────────────────────────
y_grade  = 0.0
y_hat_4  = 0.6
z_attend = -0.5
w_4      = -1.2
lr_4     = 0.1

loss_before_4 = (y_hat_4 - y_grade) ** 2
grad_4        = 2 * (y_hat_4 - y_grade) * z_attend
w_new_4       = w_4 - lr_4 * grad_4
y_hat_new_4   = w_new_4 * z_attend
loss_after_4  = (y_hat_new_4 - y_grade) ** 2

print("=" * 55)
print("EXAMPLE 4: STUDENT GRADE")
print("=" * 55)
print(f"  Error   = ŷ - y = {y_hat_4} - {y_grade} = {y_hat_4 - y_grade:.1f}  (POSITIVE — overpredicting)")
print(f"  Feature z_attend = {z_attend}  (NEGATIVE — below average)")
print(f"  Gradient = 2 × {y_hat_4 - y_grade:.1f} × {z_attend} = {grad_4:.4f}  ← NEGATIVE")
print(f"  w changes: {w_4} → {w_new_4:.4f}  (increased toward zero)")
print(f"  New prediction: {w_new_4:.4f} × {z_attend} = {y_hat_new_4:.4f}  ← lower than {y_hat_4} ✓")
print(f"  Loss:  {loss_before_4:.4f} → {loss_after_4:.4f}  (improved ✓)")
print()
print("  Tricky case! Gradient is NEGATIVE but prediction DECREASED.")
print("  Because the feature is also negative, the product works out correctly.")

---
# Example 5: Energy Consumption — Large Gradient

**Scenario:** Predict whether a building's energy usage is low (y=0) based on occupancy.

| Variable | Value | Meaning |
|---|---|---|
| `y` | 0.0 | Low energy usage expected |
| `ŷ` | 0.8 | Prediction: 80% high usage — **very wrong!** |
| `z_occ` | 2.0 | Occupancy is 2 std above average (packed building!) |
| `w` | 0.4 | Current weight for occupancy |
| `lr` | 0.1 | Learning rate |

**Key intuition:** Both the **error** and the **feature value** are large, so the gradient will be very large. This produces a **big weight update**. Notice:

`gradient = 2 × (0.8 - 0) × 2.0 = 3.2`

This is much larger than the gradient in Example 3 (-0.72). The model makes a large correction.

In [ ]:
# ─── Example 5: Energy Consumption ───────────────────────────────────────────
y_energy = 0.0
y_hat_5  = 0.8
z_occ    = 2.0
w_5      = 0.4
lr_5     = 0.1

loss_before_5 = (y_hat_5 - y_energy) ** 2
grad_5        = 2 * (y_hat_5 - y_energy) * z_occ
w_new_5       = w_5 - lr_5 * grad_5
y_hat_new_5   = w_new_5 * z_occ
loss_after_5  = (y_hat_new_5 - y_energy) ** 2

print("=" * 55)
print("EXAMPLE 5: ENERGY CONSUMPTION")
print("=" * 55)
print(f"  Error   = ŷ - y = {y_hat_5} - {y_energy} = {y_hat_5 - y_energy:.1f}  (LARGE POSITIVE)")
print(f"  Feature z_occ   = {z_occ}  (LARGE POSITIVE)")
print(f"  Gradient = 2 × {y_hat_5 - y_energy:.1f} × {z_occ} = {grad_5:.4f}  ← VERY LARGE")
print(f"  Step size = {lr_5} × {grad_5:.4f} = {lr_5 * grad_5:.4f}  ← BIG STEP")
print(f"  w: {w_5} → {w_new_5:.4f}  (decreased a lot)")
print(f"  New prediction: {w_new_5:.4f} × {z_occ} = {y_hat_new_5:.4f}  ← much lower than {y_hat_5} ✓")
print(f"  Loss:  {loss_before_5:.4f} → {loss_after_5:.4f}  (improved ✓)")
print()
print(f"  Compare gradients across examples:")
print(f"    Ex1 (loan):   {2*(y_hat_1-y_loan)*z_age:.4f}")
print(f"    Ex3 (spam):   {grad_3:.4f}")
print(f"    Ex5 (energy): {grad_5:.4f}  ← Largest! Big error × big feature")

---
# Side-by-Side: All 5 Error Landscapes

Now let's visualize all five examples on their individual error landscapes at once. Notice how the starting position, steepness, and step size differ across examples.

In [ ]:
# ─── 2×3 grid of all 5 error landscapes ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Error Landscapes — All 5 Examples', fontsize=14, fontweight='bold', y=1.01)

examples = [
    (w_1,      y_loan,   z_age,    'Ex1: Loan Approval\n(w=0.1, ŷ=0.15, y=1.0)',   'steelblue',  lr_1),
    (w_2,      y_house,  z_size,   'Ex2: House Price\n(w=0.5, ŷ=0.4, y=1.0)',       'teal',       lr_2),
    (w_3,      y_spam,   z_free,   'Ex3: Spam Detection\n(w=0.583, ŷ=0.7, y=1.0)', 'darkorange', lr_3),
    (w_4,      y_grade,  z_attend, 'Ex4: Student Grade\n(w=-1.2, ŷ=0.6, y=0.0)',   'crimson',    lr_4),
    (w_5,      y_energy, z_occ,    'Ex5: Energy Consump.\n(w=0.4, ŷ=0.8, y=0.0)',  'purple',     lr_5),
]

ax_list = axes.flatten()
for i, (w_c, y_c, z_c, title_c, color_c, lr_c) in enumerate(examples):
    plot_landscape(ax_list[i], w_c, y_c, z_c, title_c, color_c, lr_c)

# Hide the 6th unused subplot
ax_list[5].axis('off')
ax_list[5].text(0.5, 0.5, 'Gradient Sign\nTable\n(see below)',
                ha='center', va='center', fontsize=12,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

---
# Gradient Sign Interpretation Table

The direction of the weight update depends on **two things**: the sign of the error `(ŷ - y)` and the sign of the feature `z`.

| Scenario | `(ŷ - y)` | Feature `z` | Gradient | Weight update | Outcome |
|---|---|---|---|---|---|
| Underpredicting, positive feature | Negative | Positive | **Negative** | Increase `w` | Prediction goes up ✓ |
| Underpredicting, negative feature | Negative | Negative | **Positive** | Decrease `w` | Prediction goes up ✓ |
| Overpredicting, positive feature | Positive | Positive | **Positive** | Decrease `w` | Prediction goes down ✓ |
| Overpredicting, negative feature | Positive | Negative | **Negative** | Increase `w` | Prediction goes down ✓ |
| Perfect prediction | Zero | Any | **Zero** | No change | Nothing to fix ✓ |

**The beauty of gradient descent:** No matter which combination you start with, the math always moves the weight in the direction that reduces error.

Let's verify this computationally for all combinations:

In [ ]:
# ─── Verify all gradient sign combinations numerically ────────────────────────
print(f"{'Scenario':<35} {'Error':>7} {'z':>6} {'Gradient':>10} {'w change':>10} {'Correct?':>10}")
print("-" * 82)

scenarios = [
    ("Underpredict, z>0 (Ex1,Ex2)",  -0.5,  1.0),
    ("Underpredict, z<0",            -0.5, -1.0),
    ("Overpredict,  z>0 (Ex5)",       0.5,  1.0),
    ("Overpredict,  z<0 (Ex4 case)",  0.5, -1.0),
    ("Perfect prediction",            0.0,  1.0),
]

for name, err, z_val in scenarios:
    grad = 2 * err * z_val
    w_change = -0.1 * grad   # lr = 0.1
    # For underpredicting (err < 0), we want w to change so prediction goes up
    # For overpredicting  (err > 0), we want w to change so prediction goes down
    # 'goes up' means w*z increases; sign(w_change)*sign(z) > 0 for up
    if err == 0:
        correct = "No change ✓"
    elif err < 0:   # underpredicting → want prediction to go up
        up = (w_change * z_val) > 0
        correct = "Pred goes up ✓" if up else "WRONG ✗"
    else:           # overpredicting → want prediction to go down
        down = (w_change * z_val) < 0
        correct = "Pred goes down ✓" if down else "WRONG ✗"
    print(f"  {name:<33} {err:>7.1f} {z_val:>6.1f} {grad:>10.3f} {w_change:>10.4f}   {correct}")

---
# Convergence Over Iterations

So far we have seen one update at a time. In practice, gradient descent runs for **many iterations** — each time making a small correction.

Let's run **20 iterations** on Example 1 (loan approval) and watch:
1. How the **error decreases** over time
2. How the **gradient shrinks** as we approach the minimum
3. How the **weight converges** to the optimal value

In [ ]:
# ─── 20 iterations on Example 1 ──────────────────────────────────────────────
np.random.seed(42)

n_iter  = 20
y_true  = y_loan
z_feat  = z_age
lr      = lr_1
w       = w_1      # start at 0.1

history = {'iteration': [], 'w': [], 'y_hat': [], 'loss': [], 'gradient': []}

for i in range(n_iter):
    y_hat = w * z_feat
    loss  = (y_hat - y_true) ** 2
    grad  = 2 * (y_hat - y_true) * z_feat

    history['iteration'].append(i)
    history['w'].append(w)
    history['y_hat'].append(y_hat)
    history['loss'].append(loss)
    history['gradient'].append(grad)

    w = w - lr * grad   # update

# Print every 5 iterations
print(f"{'Iter':>5} {'Weight w':>10} {'Prediction ŷ':>14} {'Loss':>10} {'Gradient':>10}")
print("-" * 55)
for i in range(n_iter):
    if i % 4 == 0 or i == n_iter - 1:
        print(f"  {history['iteration'][i]:>3}   {history['w'][i]:>10.5f}   {history['y_hat'][i]:>12.5f}   "
              f"{history['loss'][i]:>9.5f}   {history['gradient'][i]:>9.5f}")
print()
print(f"  Optimal weight = y/z = {y_true}/{z_feat} = {y_true/z_feat:.5f}")
print(f"  Final weight   = {history['w'][-1]:.5f}")
print(f"  Final loss     = {history['loss'][-1]:.7f}")

In [ ]:
# ─── Convergence plots: error, gradient, weight over 20 iterations ────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Convergence Over 20 Iterations — Example 1: Loan Approval', fontsize=13, fontweight='bold')

iters = history['iteration']

# Plot 1: Loss over time
axes[0].plot(iters, history['loss'], 'o-', color='crimson', lw=2, ms=5)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss L = (ŷ - y)²')
axes[0].set_title('Loss — Should Decrease to 0')
axes[0].fill_between(iters, history['loss'], alpha=0.15, color='crimson')

# Plot 2: Gradient over time
axes[1].plot(iters, history['gradient'], 's-', color='darkorange', lw=2, ms=5)
axes[1].axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Gradient ∂L/∂w')
axes[1].set_title('Gradient — Approaches 0 at Minimum')
axes[1].fill_between(iters, history['gradient'], alpha=0.15, color='darkorange')

# Plot 3: Weight over time
w_opt = y_true / z_feat
axes[2].plot(iters, history['w'], 'D-', color='steelblue', lw=2, ms=5, label='w over time')
axes[2].axhline(w_opt, color='green', lw=1.5, linestyle='--', label=f'Optimal w = {w_opt:.3f}')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Weight w')
axes[2].set_title('Weight — Converges to Optimal')
axes[2].legend(fontsize=9)

for ax in axes:
    ax.set_xticks(range(0, n_iter, 4))

plt.tight_layout()
plt.show()

**What the convergence plots tell us:**

- **Loss plot:** Starts high (prediction far off), decreases rapidly at first, then flattens near zero. The curve is smooth because each step is proportional to the slope.

- **Gradient plot:** Starts large and negative (big uphill slope in the wrong direction), approaches zero as we near the minimum. When gradient = 0, we are at the bottom of the valley — no more updates needed.

- **Weight plot:** Starts at 0.1, increases toward the optimal value `y/z = 1.0/1.5 ≈ 0.667`. Notice it moves fast at first (large gradient) and slows down as it converges.

This is the classic convergence behaviour of gradient descent on a convex loss function.

---
# Detailed Error Landscape with Gradient Arrow

Let's draw one final detailed landscape that shows the geometry clearly: the parabola, the current position, the tangent line (representing the gradient), and the optimal point.

In [ ]:
# ─── Detailed error landscape for Example 1 with annotated tangent line ───────
fig, ax = plt.subplots(figsize=(9, 5))

w_vals = np.linspace(-0.3, 1.2, 400)
losses = (w_vals * z_age - y_loan) ** 2

ax.plot(w_vals, losses, 'steelblue', lw=2.5, label='Loss landscape L(w)')

# Mark the iteration path
path_w = np.array(history['w'][:8])
path_L = (path_w * z_age - y_loan) ** 2
ax.plot(path_w, path_L, 'o--', color='crimson', lw=1.5, ms=7, alpha=0.8, label='Gradient descent path')

for i, (pw, pl) in enumerate(zip(path_w, path_L)):
    ax.annotate(f'iter {i}', (pw, pl), textcoords='offset points',
                xytext=(5, 5), fontsize=7.5, color='crimson')

# Tangent line at starting point
w0     = history['w'][0]
L0     = (w0 * z_age - y_loan) ** 2
grad0  = history['gradient'][0]
t_range = np.linspace(w0 - 0.4, w0 + 0.4, 50)
tangent = L0 + grad0 * (t_range - w0)
ax.plot(t_range, tangent, 'darkorange', lw=1.5, linestyle=':', label=f'Tangent at w=0.1 (slope={grad0:.2f})')

# Optimal point
w_optimal = y_loan / z_age
ax.scatter([w_optimal], [0], color='green', s=120, zorder=6, marker='*', label=f'Optimal w = {w_optimal:.3f}')

# Starting point
ax.scatter([w0], [L0], color='red', s=100, zorder=6, label=f'Start w = {w0}')

ax.set_xlabel('Weight w', fontsize=12)
ax.set_ylabel('Loss L(w) = (w×z - y)²', fontsize=12)
ax.set_title('Error Landscape — Gradient Descent Path (first 8 iterations)\nExample 1: Loan Approval', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.0)

# Annotation explaining the tangent
ax.annotate('Gradient = slope of tangent\nNegative → move right (increase w)',
            xy=(w0, L0), xytext=(0.5, 0.7),
            fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.5))

plt.tight_layout()
plt.show()

---
# Summary & Key Takeaways

## What We Learned

| Concept | Formula | Intuition |
|---|---|---|
| **Loss** | `L = (ŷ - y)²` | How wrong we are, always positive |
| **Gradient** | `∂L/∂w = 2(ŷ-y)×z` | Slope of the loss landscape at current `w` |
| **Update** | `w = w - lr × gradient` | Take one step downhill |

## Five Key Insights

1. **Direction of update is automatic:** Gradient sign always pushes the weight in the right direction, regardless of whether error is positive or negative.

2. **Step size is proportional to error:** Large errors → large gradients → large steps. Small errors near the minimum → small steps (no overshooting).

3. **Feature scale matters:** Large `z` values amplify the gradient. Features far from average drive bigger weight changes.

4. **Learning rate controls speed:** Too small → slow. Too large → overshoot and diverge. We will study this deeply in Week 6-7.

5. **Convergence:** Over many iterations, both gradient and loss shrink toward zero. The weight settles at the value that produces the best predictions.

## Next Week

We have so far worked with **one weight** and **one feature**. Real models have multiple weights. In Week 6-7 we will see how gradient descent handles multiple weights simultaneously and how learning rate choice can make or break training.